# ¿Los minutos en cancha generan puntos?

### Regresión lineal múltiple y multicolinealidad — NBA 2024-25

**Proyecto de Econometría · Universidad Panamericana · Otoño 2026**

---

**Cómo usar este notebook**

1. Corre la celda 1. Te va a pedir que subas el archivo `NBA_Regresion_Completo.xlsx`
   (arrastra el archivo a la ventanita que aparece).
2. Después corre las demás celdas en orden, de arriba hacia abajo.
3. Cada bloque imprime su resultado. No hay pasos manuales entre uno y otro.

**La pregunta que contesta este análisis**

> ¿El tiempo en cancha tiene un efecto propio sobre los puntos que anota un jugador,
> o su aparente efecto es solo un reflejo de cuántos tiros toma?

La respuesta cambia por completo según qué variables metamos al modelo. Ese contraste
es el argumento del trabajo.

## 1. Preparación del entorno y carga de datos

Colab ya trae `pandas`, `numpy`, `statsmodels`, `scipy` y `matplotlib` instalados,
así que no hay que instalar nada.

La celda abre un botón para subir el Excel. Si estás corriendo esto fuera de Colab,
también funciona si el archivo está en la misma carpeta.

In [ ]:
import io, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import jarque_bera

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

ARCHIVO = "NBA_Regresion_Completo.xlsx"
HOJA = "database_24_25"

try:
    from google.colab import files        # estamos en Colab
    print("Sube el archivo", ARCHIVO, "(arrastra o selecciona):")
    subidos = files.upload()
    nombre = list(subidos.keys())[0]
    crudo = pd.read_excel(io.BytesIO(subidos[nombre]), sheet_name=HOJA)
except ImportError:                        # estamos fuera de Colab
    crudo = pd.read_excel(ARCHIVO, sheet_name=HOJA)

print(f"\nArchivo cargado: {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print("Columnas:", list(crudo.columns))
crudo.head()

## 2. Validación de los datos

Antes de limpiar nada, se comprueba que los datos sean internamente consistentes.

En el basquetbol se cumple **por definición** la siguiente identidad contable:

$$PTS = 2 \times FG + 3P + FT$$

Un tiro de campo vale 2 puntos; un triple vale 3, pero ya viene contado dentro de FG,
así que solo hay que sumarle 1 más; un tiro libre vale 1.

Si esta igualdad no se cumple en todas las filas, los datos están corruptos.
**Guarda bien este resultado: es el hallazgo que define todo el proyecto.**

In [ ]:
# Identidad contable: debe cumplirse en el 100% de las filas
identidad = 2 * crudo["FG"] + crudo["3P"] + crudo["FT"]
coincide = (identidad == crudo["PTS"]).mean()
print(f"PTS = 2*FG + 3P + FT  se cumple en {coincide:.2%} de los {len(crudo):,} registros")

# Validaciones de rango: no puede haber mas tiros anotados que intentados
print("FG <= FGA :", bool((crudo["FG"] <= crudo["FGA"]).all()))
print("3P <= 3PA :", bool((crudo["3P"] <= crudo["3PA"]).all()))
print("FT <= FTA :", bool((crudo["FT"] <= crudo["FTA"]).all()))

# Valores faltantes
faltantes = crudo.isna().sum()
print("\nColumnas con valores faltantes:",
      dict(faltantes[faltantes > 0]) or "ninguna")

## 3. Limpieza y construcción de la base de análisis

Tres decisiones que hay que poder defender:

**3.1 — La unidad de análisis es el jugador, no el partido.**
El archivo trae 16,512 registros partido a partido, pero solo 562 jugadores. Si
usáramos los 16,512 renglones estaríamos violando el supuesto de independencia de
las observaciones: cada jugador aparece hasta 50 veces y sus desempeños están
correlacionados entre sí. Eso se llama *pseudo-replicación* y hace que los errores
estándar salgan artificialmente chicos, inflando la significancia.

**3.2 — FG% se recalcula como aciertos totales / intentos totales.**
No como el promedio de los porcentajes de cada partido. Promediar razones sesga el
resultado: un partido de 1 de 1 (100%) pesaría igual que uno de 8 de 20 (40%). La
definición oficial de la NBA es la razón de totales.

**3.3 — Se filtran los jugadores con menos de 10 partidos.**
Con menos partidos, un solo juego atípico domina el promedio. El umbral de 10 es una
decisión nuestra, no un estándar de la literatura, y excluye al 15.7% de los jugadores:
por eso se reporta explícitamente.

Además, los nombres venían con la codificación dañada (`Nikola Jokiƒá` en vez de
`Nikola Jokić`), por una doble conversión UTF-8 / Mac Roman. Se repara.

In [ ]:
def reparar_acentos(texto):
    """Revierte la doble codificacion UTF-8 / Mac Roman en los nombres."""
    if not isinstance(texto, str):
        return texto
    try:
        return texto.encode("mac_roman").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto

d = crudo.copy()
d["Player"] = d["Player"].map(reparar_acentos)
d["Data"] = pd.to_datetime(d["Data"])
d = d.sort_values(["Player", "Data"])

MIN_PARTIDOS = 10
COLS = ["MP", "FGA", "3PA", "ORB", "DRB", "FTA",
        "AST", "STL", "BLK", "TOV", "PF", "PTS"]

g = d.groupby("Player")
a = g[COLS].mean()
a["FG%"] = g["FG"].sum() / g["FGA"].sum()      # razon de totales, no promedio de razones
a["GP"] = g.size()                              # partidos jugados
a["Win_Rate"] = g["Res"].mean()                 # proporcion de victorias del equipo

n_antes = len(a)
a = a[a["GP"] >= MIN_PARTIDOS].copy()
a.index.name = "Jugador"

print(f"Registros partido a partido : {len(d):,}")
print(f"Jugadores unicos            : {n_antes}")
print(f"Filtro GP >= {MIN_PARTIDOS}            : se excluyen {n_antes - len(a)} jugadores")
print(f"MUESTRA FINAL (n)           : {len(a)}")
print(f"Periodo                     : {d['Data'].min():%d-%b-%Y} a {d['Data'].max():%d-%b-%Y}")

a.sort_values("PTS", ascending=False).head()

## 4. Estadística descriptiva y correlaciones

Antes de estimar hay que conocer los datos. Dos cosas a observar:

- La distribución de PTS tiene **asimetría positiva**: la mayoría de los jugadores anota
  poco y una minoría concentra la producción ofensiva.
- La correlación bivariada entre MP y PTS es **0.878**, alta y positiva. Esa es
  justamente la correlación que el análisis multivariado va a poner a prueba.

In [ ]:
VARS = ["PTS", "MP", "FGA", "FG%", "3PA", "FTA", "ORB", "DRB",
        "AST", "STL", "BLK", "TOV", "PF", "Win_Rate"]

desc = a[VARS].describe().T[["count", "mean", "50%", "std", "min", "max"]]
desc.columns = ["n", "Media", "Mediana", "Desv.Est.", "Minimo", "Maximo"]
print("ESTADISTICAS DESCRIPTIVAS")
print(desc.round(3).to_string())

print(f"\nAsimetria de PTS = {a['PTS'].skew():.3f}")
print(f"corr(MP, PTS)    = {a['MP'].corr(a['PTS']):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(a["PTS"], bins=35, color="#2a78d6", edgecolor="white", linewidth=0.6)
ax1.axvline(a["PTS"].mean(), color="#d03b3b", ls="--", lw=1.5,
            label=f"Media = {a['PTS'].mean():.2f}")
ax1.set(xlabel="Puntos por partido", ylabel="Jugadores",
        title="Distribucion de la variable dependiente")
ax1.legend(frameon=False)

ax2.scatter(a["MP"], a["PTS"], s=18, color="#2a78d6", alpha=0.5, edgecolor="none")
b1, b0 = np.polyfit(a["MP"], a["PTS"], 1)
x = np.linspace(a["MP"].min(), a["MP"].max(), 100)
ax2.plot(x, b0 + b1 * x, color="#d03b3b", lw=1.8,
         label=f"r = {a['MP'].corr(a['PTS']):.3f}")
ax2.set(xlabel="Minutos por partido (MP)", ylabel="Puntos por partido (PTS)",
        title="Relacion bivariada minutos-puntos")
ax2.legend(frameon=False)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

plt.tight_layout(); plt.show()

## 5. El problema que define el proyecto

La identidad contable de la celda 2 dice que $PTS = 2 \times FG + 3P + FT$.
Y como $FG = FGA \times FG\%$, resulta que:

$$PTS \approx 2 \times (FGA \times FG\%) + 3P + FT$$

**Las variables FGA, FG%, 3PA y FTA son los insumos aritméticos de PTS.**

Un modelo que explique PTS con esas variables no está estimando una relación económica:
está reproduciendo una ecuación de contabilidad. Un $R^2$ cercano a 1 ahí no es evidencia
de nada, es una consecuencia mecánica de haber puesto del lado derecho los componentes de
la variable del lado izquierdo.

Por eso estimamos **dos especificaciones** y las comparamos:

| | Qué incluye | Para qué sirve |
|---|---|---|
| **Modelo 1 — contable** | FGA, FG%, 3PA, FTA | Documentar el problema. **No es el modelo preferido** |
| **Modelo 2 — rol y uso** | Solo variables de rol: minutos, asistencias, rebotes, faltas | **Especificación preferida**. Es el único que puede contestar la pregunta |

In [ ]:
X_MODELO_1 = ["MP", "FGA", "FG%", "3PA", "FTA", "ORB", "AST", "STL", "TOV", "Win_Rate"]
X_MODELO_2 = ["MP", "AST", "TOV", "ORB", "DRB", "STL", "BLK", "PF", "Win_Rate"]
ALFA = 0.05     # se fija ANTES de ver los resultados

def estimar(datos, regresoras, y="PTS"):
    """Estima por MCO. sm.add_constant agrega la columna de unos del intercepto."""
    X = sm.add_constant(datos[regresoras])
    return sm.OLS(datos[y], X).fit()

def tabla(modelo):
    """Tabla de coeficientes con error estandar, t, p, IC95%, VIF y tolerancia."""
    X = modelo.model.exog
    nombres = modelo.model.exog_names
    vifs = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    ic = modelo.conf_int(alpha=ALFA)
    t = pd.DataFrame({
        "Coef.": modelo.params, "EE": modelo.bse,
        "t": modelo.tvalues, "p": modelo.pvalues,
        "IC95_inf": ic[0], "IC95_sup": ic[1],
        "VIF": vifs,
    }, index=nombres)
    t["Tolerancia"] = 1 / t["VIF"]
    t.loc["const", ["VIF", "Tolerancia"]] = np.nan   # la constante no se interpreta
    t["Signif."] = pd.cut(t["p"], [0, .001, .01, .05, 1],
                          labels=["***", "**", "*", "n.s."])
    return t.round(4)

def encabezado(m, titulo):
    print("=" * 78); print(titulo); print("=" * 78)
    print(f"n = {int(m.nobs)}   k = {int(m.df_model)} regresoras   gl = {int(m.df_resid)}")
    print(f"R2 = {m.rsquared:.4f}   R2 ajustado = {m.rsquared_adj:.4f}   "
          f"EE residual = {np.sqrt(m.mse_resid):.4f}")
    print(f"F global = {m.fvalue:.2f}   p = {m.f_pvalue:.3e}\n")

m1 = estimar(a, X_MODELO_1)
encabezado(m1, "MODELO 1 - especificacion contable (NO preferida)")
print(tabla(m1).to_string())

## 6. Modelo 2 — especificación preferida

Este modelo excluye todo componente de la identidad contable. Explica los puntos a
partir del **rol** del jugador en la cancha, no de sus propios tiros.

Signos esperados, planteados **antes** de estimar:

| Variable | Signo esperado | Por qué |
|---|---|---|
| MP | **+** | Más tiempo en cancha, más oportunidades de anotar |
| TOV | **+** | Las pérdidas señalan alto uso del balón (no causan puntos) |
| PF | **−** | Las faltas limitan el tiempo efectivo en cancha |
| STL, BLK | **−** | Especialización defensiva |
| Win_Rate | **+** | Mejor equipo, más oportunidades ofensivas |
| AST, ORB | ambiguo | Puede ser pasador o anotador; rol interior |

In [ ]:
m2 = estimar(a, X_MODELO_2)
encabezado(m2, "MODELO 2 - rol y uso (ESPECIFICACION PREFERIDA)")
print(tabla(m2).to_string())

## 7. Pruebas de hipótesis y comparación

La prueba central del proyecto:

$$H_0: \beta_{MP} = 0 \qquad\text{contra}\qquad H_1: \beta_{MP} \neq 0$$

Es **bilateral** (el signo ≠ en H₁), al nivel $\alpha = 0.05$. Con $\nu = n - k - 1$
grados de libertad y $n = 474$, el valor crítico de la $t$ es aproximadamente 1.965.

Y una prueba **F conjunta** sobre el bloque contable del Modelo 1:

$$H_0: \beta_{FGA} = \beta_{FG\%} = \beta_{3PA} = \beta_{FTA} = 0$$

Son $q = 4$ restricciones simultáneas. Una prueba F siempre es de **una sola cola**
(la derecha), porque el numerador nunca puede ser negativo.

In [ ]:
print("PRUEBA t:   H0: beta_MP = 0   vs   H1: beta_MP != 0   (bilateral, alfa = 0.05)")
print("-" * 78)
for nombre, m in [("Modelo 1", m1), ("Modelo 2", m2)]:
    b, ee = m.params["MP"], m.bse["MP"]
    t, p = m.tvalues["MP"], m.pvalues["MP"]
    inf, sup = m.conf_int(alpha=ALFA).loc["MP"]
    critico = st.t.ppf(1 - ALFA / 2, m.df_resid)
    decision = "SE RECHAZA H0" if abs(t) > critico else "NO se rechaza H0"
    print(f"{nombre}: b = {b:+.4f}  EE = {ee:.4f}  t = {t:+.3f}  "
          f"|t| vs t_critico({critico:.3f})  p = {p:.4f}")
    print(f"          IC95% = [{inf:+.4f}, {sup:+.4f}]  ->  {decision}\n")

# Prueba F conjunta del bloque contable en el Modelo 1
r = m1.f_test([f"{v} = 0" for v in ["FGA", "FG%", "3PA", "FTA"]])
print(f"PRUEBA F conjunta  H0: beta_FGA = beta_FG% = beta_3PA = beta_FTA = 0   (q = 4)")
print(f"F({4}, {int(m1.df_resid)}) = {float(np.squeeze(r.fvalue)):.2f}   "
      f"p = {float(np.squeeze(r.pvalue)):.3e}")
print("-> el bloque contable domina el ajuste del Modelo 1\n")

t1, t2 = tabla(m1), tabla(m2)
comp = pd.DataFrame({
    "Modelo 1 (contable)": [
        int(m1.nobs), int(m1.df_model), round(m1.rsquared, 4), round(m1.rsquared_adj, 4),
        round(m1.fvalue, 1), f"{m1.params['MP']:+.4f}", round(m1.tvalues["MP"], 3),
        round(m1.pvalues["MP"], 4), "NO se rechaza H0",
        f"{t1['VIF'].max():.2f} ({t1['VIF'].idxmax()})"],
    "Modelo 2 (rol y uso)": [
        int(m2.nobs), int(m2.df_model), round(m2.rsquared, 4), round(m2.rsquared_adj, 4),
        round(m2.fvalue, 1), f"{m2.params['MP']:+.4f}", round(m2.tvalues["MP"], 3),
        round(m2.pvalues["MP"], 4), "SE RECHAZA H0",
        f"{t2['VIF'].max():.2f} ({t2['VIF'].idxmax()})"],
}, index=["n", "k", "R2", "R2 ajustado", "F global", "beta de MP", "t de MP",
          "p de MP", "Decision sobre MP", "VIF maximo"])

print("=" * 78); print("COMPARACION DE ESPECIFICACIONES"); print("=" * 78)
print(comp.to_string())
print("\nLa misma pregunta recibe respuestas OPUESTAS segun la especificacion.")
print("El R2 mas alto corresponde al modelo menos informativo.")

## 8. Multicolinealidad — las cuatro piezas

Las notas del curso son explícitas en que **no existe una sola "tabla de
multicolinealidad"**. Hay que leer cuatro piezas en conjunto:

**(i) Matriz de correlaciones.** Detecta relaciones lineales entre *pares* de regresoras.
Es la primera inspección, pero no detecta dependencias que involucren tres o más variables.

**(ii) VIF y tolerancia.** Para cada regresora $X_j$ se corre una regresión auxiliar de
$X_j$ sobre todas las demás regresoras, se obtiene su $R_j^2$, y entonces:

$$VIF_j = \frac{1}{1 - R_j^2} \qquad\qquad Tol_j = 1 - R_j^2 = \frac{1}{VIF_j}$$

**(iii) Autovalores e índices de condición.** Después de estandarizar las columnas de X,
los autovalores pequeños señalan direcciones de poca variación:

$$\kappa_j = \sqrt{\frac{\lambda_{máx}}{\lambda_j}}$$

**(iv) Número de condición.** Resume qué tan cerca está la matriz de diseño de una
dependencia lineal problemática. Depende de la escala, por eso se calcula sobre la
matriz estandarizada.

> **Umbrales: orientación, no ley.** VIF > 5 o VIF > 10 son señales de atención, no
> teoremas. La pregunta relevante es si la imprecisión resultante impide contestar la
> pregunta económica que motivó el modelo.

In [ ]:
def diagnostico_colinealidad(datos, regresoras, etiqueta):
    print("=" * 78); print(f"MULTICOLINEALIDAD - {etiqueta}"); print("=" * 78)

    # (i) correlaciones entre regresoras
    c = datos[regresoras].corr()
    pares = (c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
              .stack().abs().sort_values(ascending=False))
    print("(i) Pares de regresoras con mayor correlacion absoluta:")
    for (v1, v2), r in pares.head(4).items():
        print(f"      corr({v1}, {v2}) = {c.loc[v1, v2]:+.4f}")

    # (ii) VIF y tolerancia, calculados a mano para mostrar de donde salen
    print("\n(ii) VIF y tolerancia (de la regresion auxiliar de cada Xj):")
    filas = []
    for v in regresoras:
        otras = [u for u in regresoras if u != v]
        r2_aux = sm.OLS(datos[v], sm.add_constant(datos[otras])).fit().rsquared
        filas.append((v, r2_aux, 1 / (1 - r2_aux), 1 - r2_aux))
    vt = pd.DataFrame(filas, columns=["Variable", "R2_auxiliar", "VIF", "Tolerancia"])
    vt["Alerta"] = np.where(vt["VIF"] > 10, "VIF > 10",
                   np.where(vt["VIF"] > 5, "VIF > 5", ""))
    print(vt.round(4).to_string(index=False))

    # (iii) y (iv) autovalores, indices de condicion y numero de condicion
    Z = (datos[regresoras] - datos[regresoras].mean()) / datos[regresoras].std()
    lambdas = np.linalg.eigvalsh(np.corrcoef(Z.values, rowvar=False))[::-1]
    kappas = np.sqrt(lambdas.max() / lambdas)
    print("\n(iii) Autovalores de la matriz de correlaciones e indices de condicion:")
    print(pd.DataFrame({"Autovalor": lambdas.round(4),
                        "Indice de condicion": kappas.round(3)}).to_string(index=False))
    print(f"\n(iv) Numero de condicion (kappa maximo) = {kappas.max():.2f}"
          f"   ({'ALERTA: supera 30' if kappas.max() > 30 else 'por debajo de 30'})\n")
    return vt

vt1 = diagnostico_colinealidad(a, X_MODELO_1, "Modelo 1 (contable)")
vt2 = diagnostico_colinealidad(a, X_MODELO_2, "Modelo 2 (rol y uso)")

## 9. Diagnóstico de los supuestos

Se evalúa el modelo preferido contra los supuestos del modelo lineal clásico:

| Supuesto | Prueba | H₀ |
|---|---|---|
| Forma funcional | RESET de Ramsey | La especificación lineal es adecuada |
| Homocedasticidad | Breusch-Pagan, White | La varianza del error es constante |
| Normalidad | Jarque-Bera, Shapiro-Wilk | Los residuos son normales |

**Importante para la exposición:** varios de estos supuestos se rechazan. La respuesta
correcta no es esconderlo ni presentar una corrección como solución automática, sino
acotar qué significa cada rechazo. Las notas del curso insisten en esto.

In [ ]:
resid, exog = m2.resid, m2.model.exog
filas = []

reset = linear_reset(m2, power=2, test_type="fitted", use_f=True)
filas.append(("Forma funcional", "RESET de Ramsey",
              f"F = {float(reset.statistic):.3f}", float(reset.pvalue)))
bp = het_breuschpagan(resid, exog)
filas.append(("Homocedasticidad", "Breusch-Pagan", f"LM = {bp[0]:.3f}", bp[1]))
wh = het_white(resid, exog)
filas.append(("Homocedasticidad", "White", f"LM = {wh[0]:.3f}", wh[1]))
jb, jb_p, sesgo, curt = jarque_bera(resid)
filas.append(("Normalidad", "Jarque-Bera", f"JB = {jb:.3f}", jb_p))
sw, sw_p = st.shapiro(resid)
filas.append(("Normalidad", "Shapiro-Wilk", f"W = {sw:.4f}", sw_p))

sup = pd.DataFrame(filas, columns=["Supuesto", "Prueba", "Estadistico", "p"])
sup["Decision al 5%"] = np.where(sup["p"] < ALFA, "Se rechaza H0", "No se rechaza H0")
sup["p"] = sup["p"].map(lambda v: f"{v:.4f}" if v >= 1e-4 else f"{v:.2e}")
print("PRUEBAS FORMALES DE SUPUESTOS"); print(sup.to_string(index=False))
print(f"\nResiduos: asimetria = {sesgo:.3f}, curtosis = {curt:.3f}")

# Cuatro paneles de diagnostico visual
inf_obj = m2.get_influence()
d_res = pd.DataFrame({
    "ajustado": m2.fittedvalues, "residual": m2.resid,
    "estandarizado": inf_obj.resid_studentized_internal,
    "apalancamiento": inf_obj.hat_matrix_diag,
    "cook": inf_obj.cooks_distance[0]}, index=a.index)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0,0].scatter(d_res["ajustado"], d_res["residual"], s=14, color="#2a78d6", alpha=.5)
ax[0,0].axhline(0, color="#d03b3b", lw=1.2)
ax[0,0].set(xlabel="Valores ajustados", ylabel="Residuos",
            title="Residuos vs ajustados (se busca nube sin patron)")
st.probplot(d_res["estandarizado"], dist="norm", plot=ax[0,1])
ax[0,1].get_lines()[0].set(marker="o", ms=3, color="#2a78d6", alpha=.6, mec="none")
ax[0,1].get_lines()[1].set(color="#d03b3b", lw=1.2)
ax[0,1].set_title("Q-Q de residuos estandarizados (normalidad)")
ax[1,0].hist(d_res["residual"], bins=35, color="#2a78d6", ec="white", lw=.5, density=True)
xx = np.linspace(d_res["residual"].min(), d_res["residual"].max(), 200)
ax[1,0].plot(xx, st.norm.pdf(xx, d_res["residual"].mean(), d_res["residual"].std()),
             color="#d03b3b", lw=1.4)
ax[1,0].set(xlabel="Residuos", ylabel="Densidad", title="Distribucion de los residuos")
ax[1,1].scatter(d_res["apalancamiento"], d_res["estandarizado"],
                s=20 + 900 * d_res["cook"] / d_res["cook"].max(),
                color="#2a78d6", alpha=.4, ec="none")
for lim in (-3, 0, 3):
    ax[1,1].axhline(lim, color="#d03b3b" if lim else "#898781", lw=.8,
                    ls="--" if lim else "-")
ax[1,1].set(xlabel="Apalancamiento", ylabel="Residuo estandarizado",
            title="Influencia (tamano = distancia de Cook)")
for e in ax.flat:
    e.spines[["top", "right"]].set_visible(False); e.grid(alpha=.3); e.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 10. Robustez

Tres verificaciones de que la conclusión sobre MP no depende de decisiones arbitrarias:

1. **Errores estándar robustos HC3**, por la heterocedasticidad detectada. Lo que importa
   no es que los errores cambien, sino si cambia alguna *decisión* de significancia.
2. **Forma funcional log-nivel**, por el rechazo de RESET. En esta forma,
   $100 \times \beta$ se lee como cambio porcentual.
3. **Exclusión de observaciones influyentes**, para verificar que el resultado no depende
   de un puñado de superestrellas.

In [ ]:
# 1) Errores estandar robustos HC3
rob = m2.get_robustcov_results(cov_type="HC3")
comp_rob = pd.DataFrame({
    "Coef.": m2.params, "EE convencional": m2.bse, "EE robusto HC3": rob.bse,
    "t convencional": m2.tvalues, "t robusto": rob.tvalues, "p robusto": rob.pvalues,
}, index=m2.model.exog_names).round(4)
cambia = (m2.pvalues < ALFA).values != (pd.Series(rob.pvalues) < ALFA).values
comp_rob["Cambia decision"] = np.where(cambia, "SI", "no")
print("1) ERRORES ESTANDAR ROBUSTOS (HC3)"); print(comp_rob.to_string())
print(f"\n   Decisiones de significancia que cambian: {cambia.sum()} de {len(cambia)}\n")

# 2) Forma funcional log-nivel
a_pos = a[a["PTS"] > 0]
m3 = sm.OLS(np.log(a_pos["PTS"]), sm.add_constant(a_pos[X_MODELO_2])).fit()
print("2) FORMA LOG-NIVEL")
print(f"   n = {int(m3.nobs)}   R2 = {m3.rsquared:.4f}")
print(f"   beta_MP = {m3.params['MP']:+.4f}  (t = {m3.tvalues['MP']:.2f})  ->  "
      f"un minuto adicional se asocia con {100*m3.params['MP']:.2f}% mas puntos\n")

# 3) Exclusion de observaciones influyentes
n, k = int(m2.nobs), int(m2.df_model) + 1
umbral_apal, umbral_cook = 2 * k / n, 4 / n
marcadas = d_res[(d_res["apalancamiento"] > umbral_apal) | (d_res["cook"] > umbral_cook)]
m_sin = estimar(a.drop(index=marcadas.index), X_MODELO_2)
cambio = abs(m_sin.params["MP"] - m2.params["MP"]) / abs(m2.params["MP"])
print("3) OBSERVACIONES INFLUYENTES")
print(f"   Umbrales: apalancamiento > 2k/n = {umbral_apal:.4f}, Cook > 4/n = {umbral_cook:.4f}")
print(f"   Marcadas: {len(marcadas)} de {n} ({len(marcadas)/n:.1%})")
print(f"   Mas influyentes: {', '.join(marcadas.nlargest(3, 'cook').index)}")
print(f"   beta_MP completo = {m2.params['MP']:+.4f}   "
      f"sin influyentes = {m_sin.params['MP']:+.4f}   (cambio de {cambio:.1%})")

## 11. Conclusión

**Respuesta a la pregunta.** Una vez que se excluyen los componentes aritméticos de la
variable dependiente, el tiempo en cancha **sí tiene una asociación propia, positiva y
estadísticamente significativa** con los puntos anotados: **+0.477 puntos por minuto
adicional**, con un intervalo de confianza del 95% de [0.423, 0.532].

**El hallazgo metodológico.** Una sola decisión de modelado —incluir o excluir los
componentes de la identidad contable— mueve $\hat\beta_{MP}$ de +0.002 ($t = 0.27$,
no significativo) a +0.477 ($t = 17.21$, $p < 0.001$). **El modelo con $R^2 = 0.99$ es el
que menos responde la pregunta de investigación.**

### Límites del análisis

1. **No se identifica causalidad.** Los minutos no se asignan al azar: el entrenador da
   más tiempo a quien ya anota bien. La causalidad corre en ambos sentidos y este diseño
   observacional no las separa.
2. **Variables omitidas.** No se observan posición, talento previo, sistema ofensivo ni
   lesiones. Todo eso queda en el término de error y está correlacionado con MP, lo que
   compromete el supuesto $E(\varepsilon \mid X) = 0$.
3. **Forma funcional.** RESET rechaza la linealidad tanto en niveles como en log-nivel.
   Hay curvatura que el modelo no captura.
4. **TOV es un proxy imperfecto** del uso del balón. Su coeficiente positivo no significa
   que perder el balón produzca puntos.
5. **Periodo incompleto.** La muestra llega al 7 de febrero de 2025, aproximadamente dos
   tercios de la temporada regular.

### ¿Qué decisión de modelado podría cambiar la conclusión?

Incluir los componentes de la identidad contable. Es la única decisión que, por sí sola,
invierte la respuesta a la pregunta de investigación.

In [ ]:
print("RESUMEN FINAL")
print("=" * 60)
print(f"Muestra              : n = {int(m2.nobs)} jugadores, temporada NBA 2024-25")
print(f"Modelo preferido     : PTS ~ {' + '.join(X_MODELO_2)}")
print(f"R2                   : {m2.rsquared:.4f}")
print(f"F global             : {m2.fvalue:.1f} (p = {m2.f_pvalue:.2e})")
print(f"beta_MP              : {m2.params['MP']:+.4f} puntos por minuto adicional")
print(f"IC 95% de beta_MP    : [{m2.conf_int().loc['MP', 0]:+.4f}, "
      f"{m2.conf_int().loc['MP', 1]:+.4f}]")
print(f"t de MP              : {m2.tvalues['MP']:.2f}  ->  se rechaza H0 al 5%")
print("=" * 60)